In [1]:
#load packages
import os
import numpy as np
import pandas as pd


In [10]:
#import data
file_path = "/group/moniergrp/dbaral/run_project/input_data"
save_path = "/group/moniergrp/dbaral/run_project/input_data/model_input"

yield_df = pd.read_csv(file_path + '/yield/rice_yield_1979_2023.csv')
stress = pd.read_csv(file_path + '/gridmet/AnnualStress_1979_2023_v2.csv')
booting = pd.read_csv(file_path + '/gridmet/booting_feature_variables.csv')
flowering = pd.read_csv(file_path + '/gridmet/flowering_feature_variables.csv')
grainfill =  pd.read_csv(file_path + '/gridmet/grainfill_feature_variables.csv')
averageTemp = pd.read_csv(file_path + '/gridmet/growing_season_feature_variables.csv')

In [11]:
print(yield_df.head())
print(stress.head())
print(booting.head())
print(flowering.head())
print(grainfill.head())
print(averageTemp.head())

  county  year  yield_kg_ha
0  Butte  2022   10323.0285
1  Butte  2021   10524.7815
2  Butte  2020   10457.5305
3  Butte  2019    9818.6460
4  Butte  2016   10020.3990
   county  year  cdstress_bo  cdstress_fl  htstress_fl
0    Yuba  1979     4.557548          0.0          0.0
1  Colusa  1979    13.011143          0.0          0.0
2   Butte  1979    12.659294          0.0          0.0
3   Glenn  1979    14.016771          0.0          0.0
4  Placer  1979     9.033151          0.0          0.0
  county  year    tmmn_bo    tmmx_bo   tmean_bo
0  Butte  1979  14.028585  32.290268  23.159426
1  Butte  1980  16.351024  36.064868  26.207946
2  Butte  1981  14.351775  34.420294  24.386035
3  Butte  1982  14.457204  32.449794  23.453499
4  Butte  1983  15.304734  32.624651  23.964693
  county  year    tmmn_fl    tmmx_fl   tmean_fl
0  Butte  1979  18.992991  37.901827  28.447409
1  Butte  1980  17.603108  37.554250  27.578679
2  Butte  1981  16.029786  37.213902  26.621844
3  Butte  1982  15.734

In [12]:
#Stress df has repeated years columns - each year repeating thrice for booting, flowering and grain_fill period
#group the data by couunty and year and sum the stress columns to get one column for all stress indices for one year

# Group by county and year and sum the stress columns
stress = stress.groupby(['county', 'year'])[['cdstress_bo', 'cdstress_fl', 'htstress_fl']].sum().reset_index()

print(stress.head())

  county  year  cdstress_bo  cdstress_fl  htstress_fl
0  Butte  1979    12.659294     0.000000    15.958686
1  Butte  1980     4.333647     0.000000    22.792180
2  Butte  1981    11.451795     0.000000    11.827509
3  Butte  1982     8.118769     0.000000     3.126701
4  Butte  1983     9.187505     0.696218     0.006399


In [13]:
#Merge all datasets

model_df = pd.merge(
    left= yield_df,
    right=stress,
    how='outer',
    on=['county','year']
)

model_df = pd.merge(
    left=model_df,
    right=booting,
    how='outer',
    on=['county','year']
)

model_df = pd.merge(
    left=model_df,
    right=flowering,
    how='outer',
    on=['county','year']
)

model_df = pd.merge(
    left=model_df,
    right=grainfill,
    how='outer',
    on=['county','year']
)

lasso_data = pd.merge(
    left=model_df,
    right=averageTemp,
    how='outer',
    on=['county','year']
)
print(lasso_data.head())

export = lasso_data.to_csv(save_path + '/Lasso_Model_Input_Variables_1979_2023_v2.csv', index=False)


  county  year  yield_kg_ha  cdstress_bo  cdstress_fl  htstress_fl    tmmn_bo  \
0  Butte  1979    7442.4440    12.659294     0.000000    15.958686  14.028585   
1  Butte  1980    7341.5675     4.333647     0.000000    22.792180  16.351024   
2  Butte  1981    7621.7800    11.451795     0.000000    11.827509  14.351775   
3  Butte  1982    7397.6100     8.118769     0.000000     3.126701  14.457204   
4  Butte  1983    7857.1585     9.187505     0.696218     0.006399  15.304734   

     tmmx_bo   tmean_bo    tmmn_fl    tmmx_fl   tmean_fl    tmmn_gf  \
0  32.290268  23.159426  18.992991  37.901827  28.447409  14.858504   
1  36.064868  26.207946  17.603108  37.554250  27.578679  13.414532   
2  34.420294  24.386035  16.029786  37.213902  26.621844  15.016786   
3  32.449794  23.453499  15.734922  35.508514  25.621718  15.456566   
4  32.624651  23.964693  14.771091  33.263508  24.017300  16.813893   

     tmmx_gf   tmean_gf    tmmn_gs    tmmx_gs   tmean_gs  
0  34.227911  24.543207  15